# dice (Dice)

In [ ]:
///- --test static

open testing

In [ ]:
open sm'_operators

## sixth_power_sequence

In [ ]:
inl sixth_power_sequence () =
    stream.iterate_map ((*) 6) (fun x => if x <= 0 then None else Some x) 1

In [1]:
///- --test

sixth_power_sequence ()
|> stream.take_while (fun _ i => i <= 7i32)
|> stream.to_list
|> _assert_eq [ 1u64; 6; 36; 216; 1296; 7776; 46656; 279936 ]

__assert_eq { expected = Cons(1, Cons(6, Cons(36, Cons(216, Cons(1296, Cons(7776, Cons(46656, Cons(279936, Nil)))))))) }

## accumulate_dice_rolls

In [ ]:
inl accumulate_dice_rolls
    (rolls : list u8)
    (power : i8)
    acc
    =
    inl rec body forall a {number} b {number} c {number}.
        (acc : a)
        (power : b)
        (rolls : list c)
        : option (a * list c)
        =
        match rolls with
        | _ when power < 0 =>
            inl result = acc + 1
            trace Debug
                fun () => "dice.accumulate_dice_rolls"
                fun () => { power acc result }
            Some (result, rolls)
        | [] => None
        | roll :: rest when roll > 1 =>
            inl coeff = sixth_power_sequence () |> stream.item power
            inl value = conv (roll - 1) * coeff
            trace Debug
                fun () => "dice.accumulate_dice_rolls"
                fun () => { power acc roll value }
            루프 (acc + value) (power - 1) rest
        | roll :: rest =>
            trace Debug
                fun () => "dice.accumulate_dice_rolls"
                fun () => { power acc roll }
            루프 acc (power - 1) rest
    and inl 루프 forall a {number} b {number} c {number}.
        (acc : a)
        (power : b)
        (rolls : list c)
        : option (a * list c)
        =
        if var_is rolls |> not
        then body acc power rolls
        else
            inl acc = dyn acc
            join body acc power rolls
    루프 acc power rolls

In [2]:
///- --test

accumulate_dice_rolls [ 6; 5; 4; 3; 2 ] 0 1000i32
|> _assert_eq (Some (1006, [ 5; 4; 3; 2 ]))

16:36:02 1d dice.accumulate_dice_rolls / { power = 0; acc = 1000; roll = 6; value = 5 }
16:36:02 2d dice.accumulate_dice_rolls / { power = -1; acc = 1005; result = 1006 }
__assert_eq { expected = Some(1006, Cons(5, Cons(4, Cons(3, Cons(2, Nil))))) }

In [3]:
///- --test

accumulate_dice_rolls [ 6; 5; 4; 3; 2 ] 1 1000i32
|> _assert_eq (Some (1035, [ 4; 3; 2 ]))

16:36:03 1d dice.accumulate_dice_rolls / { power = 1; acc = 1000; roll = 6; value = 30 }
16:36:03 2d dice.accumulate_dice_rolls / { power = 0; acc = 1030; roll = 5; value = 4 }
16:36:03 3d dice.accumulate_dice_rolls / { power = -1; acc = 1034; result = 1035 }
__assert_eq { expected = Some(1035, Cons(4, Cons(3, Cons(2, Nil)))) }

In [4]:
///- --test

accumulate_dice_rolls [ 6; 5; 4; 3; 2 ] 2 1000i32
|> _assert_eq (Some (1208, [ 3; 2 ]))

16:36:03 1d dice.accumulate_dice_rolls / { power = 2; acc = 1000; roll = 6; value = 180 }
16:36:03 2d dice.accumulate_dice_rolls / { power = 1; acc = 1180; roll = 5; value = 24 }
16:36:03 3d dice.accumulate_dice_rolls / { power = 0; acc = 1204; roll = 4; value = 3 }
16:36:03 4d dice.accumulate_dice_rolls / { power = -1; acc = 1207; result = 1208 }
__assert_eq { expected = Some(1208, Cons(3, Cons(2, Nil))) }

## roll_within_bounds

In [ ]:
inl roll_within_bounds max rolls =
    inl power = listm.length rolls - 1
    match accumulate_dice_rolls rolls power 0 with
    | Some (result, _) when result >= 1 && result <= max => Some result
    | _ => None

In [5]:
///- --test

roll_within_bounds 2000i32 [ 1; 5; 4; 4; 5 ]
|> _assert_eq (Some 995)

16:36:08 1d dice.accumulate_dice_rolls / { power = 4; acc = 0; roll = 1 }
16:36:08 2d dice.accumulate_dice_rolls / { power = 3; acc = 0; roll = 5; value = 864 }
16:36:08 3d dice.accumulate_dice_rolls / { power = 2; acc = 864; roll = 4; value = 108 }
16:36:08 4d dice.accumulate_dice_rolls / { power = 1; acc = 972; roll = 4; value = 18 }
16:36:08 5d dice.accumulate_dice_rolls / { power = 0; acc = 990; roll = 5; value = 4 }
16:36:08 6d dice.accumulate_dice_rolls / { power = -1; acc = 994; result = 995 }
__assert_eq { expected = Some(995) }

In [6]:
///- --test

roll_within_bounds 2000i32 [ 2; 2; 6; 4; 5 ]
|> _assert_eq (Some 1715)

16:36:07 1d dice.accumulate_dice_rolls / { power = 4; acc = 0; roll = 2; value = 1296 }
16:36:07 2d dice.accumulate_dice_rolls / { power = 3; acc = 1296; roll = 2; value = 216 }
16:36:07 3d dice.accumulate_dice_rolls / { power = 2; acc = 1512; roll = 6; value = 180 }
16:36:07 4d dice.accumulate_dice_rolls / { power = 1; acc = 1692; roll = 4; value = 18 }
16:36:07 5d dice.accumulate_dice_rolls / { power = 0; acc = 1710; roll = 5; value = 4 }
16:36:07 6d dice.accumulate_dice_rolls / { power = -1; acc = 1714; result = 1715 }
__assert_eq { expected = Some(1715) }

In [7]:
///- --test

roll_within_bounds 2000i32 [ 4; 1; 1; 2; 3 ]
|> _assert_eq None

16:36:08 1d dice.accumulate_dice_rolls / { power = 4; acc = 0; roll = 4; value = 3888 }
16:36:08 2d dice.accumulate_dice_rolls / { power = 3; acc = 3888; roll = 1 }
16:36:08 3d dice.accumulate_dice_rolls / { power = 2; acc = 3888; roll = 1 }
16:36:08 4d dice.accumulate_dice_rolls / { power = 1; acc = 3888; roll = 2; value = 6 }
16:36:08 5d dice.accumulate_dice_rolls / { power = 0; acc = 3894; roll = 3; value = 2 }
16:36:08 6d dice.accumulate_dice_rolls / { power = -1; acc = 3896; result = 3897 }
__assert_eq { expected = None }

## calculate_dice_count

In [ ]:
inl calculate_dice_count forall a {number} b {number}. (max : a) : b =
    inl rec body forall t {number}. (n : t) (p : a) : t =
        inl return () =
            trace Debug
                fun () => "dice.calculate_dice_count"
                fun () => { max p n }
            n
        if p < max then
            if p > (limit.max : a) / 6
            then failwith ("dice.calculate_dice_count / max: " ++# #?max ++# " is above the largest supported bound " ++# #?p)
            else 루프 (n + 1) (p * 6)
        else return ()
    and inl 루프 forall t {number}. (n : t) (p : a) : t =
        if var_is max |> not
        then body n p
        else
            inl n = dyn n
            inl p = dyn p
            join body n p
    if max = 1
    then 1
    else 루프 0 1

In [8]:
///- --test

calculate_dice_count 36i32
|> _assert_eq 2i32

16:36:09 1d dice.calculate_dice_count / { max = 36; p = 36; n = 2 }
__assert_eq { expected = 2 }

In [9]:
///- --test

calculate_dice_count 7777i32
|> _assert_eq 6i32

16:36:10 1d dice.calculate_dice_count / { max = 7777; p = 46656; n = 6 }
__assert_eq { expected = 6 }

In [10]:
///- --test

calculate_dice_count 4738381338321616896u64
|> _assert_eq 24i32

16:36:10 1d dice.calculate_dice_count / { max = 4738381338321616896; p = 4738381338321616896; n = 24 }
__assert_eq { expected = 24 }

In [11]:
///- --test

calculate_dice_count 362797056i32
|> _assert_eq 11i32

16:36:11 1d dice.calculate_dice_count / { max = 362797056; p = 362797056; n = 11 }
__assert_eq { expected = 11 }

In [12]:
///- --test
///> rust

fun () =>
    (calculate_dice_count 4738381338321616897u64 : i32)
    |> ignore
|> _throws
|> optionm.map (sm'.format_exception >> sm'.contains "is above the largest supported bound 4738381338321616896")
|> _assert_eq (Some true)

__assert_eq { expected = Some(true) }

0

In [13]:
///- --test
///> rust

fun () =>
    (calculate_dice_count (limit.max : i64) : i32)
    |> ignore
|> _throws
|> optionm.map (sm'.format_exception >> sm'.contains "is above the largest supported bound 4738381338321616896")
|> _assert_eq (Some true)

__assert_eq { expected = Some(true) }

0

In [14]:
///- --test
///> rust

fun () =>
    (calculate_dice_count 362797057i32 : i32)
    |> ignore
|> _throws
|> optionm.map (sm'.format_exception >> sm'.contains "is above the largest supported bound 362797056")
|> _assert_eq (Some true)

__assert_eq { expected = Some(true) }

0

## roll_dice

In [ ]:
let roll_dice () : u8 =
    random' 1 7

## rotate_number

In [ ]:
inl rotate_number forall t {number}. (max : i64) (n : t) : t =
    (conv n - 1 + max) % max + 1 |> conv

## rotate_numbers

In [ ]:
inl rotate_numbers max items =
    items |> stream.map (rotate_number max)

In [15]:
///- --test

listm'.init_series -1 14 1i32
|> stream.from_list
|> rotate_numbers 6
|> stream.to_list
|> _assert_eq [ 5; 6; 1; 2; 3; 4; 5; 6; 1; 2; 3; 4; 5; 6; 1; 2 ]

__assert_eq { expected = Cons(5, Cons(6, Cons(1, Cons(2, Cons(3, Cons(4, Cons(5, Cons(6, Cons(1, Cons(2, Cons(3, Cons(4, Cons(5, Cons(6, Cons(1, Cons(2, Nil)))))))))))))))) }

## create_sequential_roller

In [ ]:
inl create_sequential_roller list =
    inl s = list |> listm.rev |> listm.append list |> stream.from_list |> stream.memoize
    inl current_index = mut 0i64
    inl acc = mut 1
    inl len = mut -1
    inl last_item = mut None
    let rec 루프 () =
        fun () =>
            inl current_index = *current_index
            inl acc = *acc
            inl len = *len
            inl last_item = *last_item |> optionm'.box
            trace Debug
                fun () => "dice.create_sequential_roller / roll"
                fun () => { current_index acc len last_item = #?*last_item }
        |> fun x => x ()
        match s () |> stream.try_item *current_index with
        | Some item =>
            current_index <- *current_index + 1
            last_item <- Some item
            item
        | None =>
            trace Debug (fun () => "dice.create_sequential_roller / roll / None") id
            if *len = -1
            then len <- *current_index
            acc <-
                if *acc >= *len
                then 1
                else *acc + 1
            current_index <- *acc - 1
            last_item <- None
            루프 ()
    루프

In [16]:
///- --test

inl sequential_roll = create_sequential_roller [ 1i32; 2; 3; 4 ]

am.init 50i32 (ignore >> sequential_roll)
|> _assert_eq (a ;[ 1; 2; 3; 4; 4; 3; 2; 1; 2; 3; 4; 4; 3; 2; 1; 3; 4; 4; 3; 2; 1; 4; 4; 3; 2; 1;
4; 3; 2; 1; 3; 2; 1; 2; 1; 1; 1; 2; 3; 4; 4; 3; 2; 1; 2; 3; 4; 4; 3; 2 ] : a i32 i32)

16:36:23 1d dice.create_sequential_roller / roll / { current_index = 0; acc = 1; len = -1; last_item = null }
16:36:23 2d dice.create_sequential_roller / roll / { current_index = 1; acc = 1; len = -1; last_item = 1 }
16:36:23 3d dice.create_sequential_roller / roll / { current_index = 2; acc = 1; len = -1; last_item = 2 }
16:36:23 4d dice.create_sequential_roller / roll / { current_index = 3; acc = 1; len = -1; last_item = 3 }
16:36:23 5d dice.create_sequential_roller / roll / { current_index = 4; acc = 1; len = -1; last_item = 4 }
16:36:23 6d dice.create_sequential_roller / roll / { current_index = 5; acc = 1; len = -1; last_item = 4 }
16:36:23 7d dice.create_sequential_roller / roll / { current_index = 6; acc = 1; len = -1; last_item = 3 }
16:36:23 8d dice.create_sequential_roller / roll / { current_index = 7; acc = 1; len = -1; last_item = 2 }
16:36:23 9d dice.create_sequential_roller / roll / { current_index = 8; acc = 1; len = -1; last_item = 1 }
16:36:23 10d dice.create_sequentia

## roll_progressively

In [ ]:
inl roll_progressively roll reroll max =
    inl power = (calculate_dice_count max) - 1
    let rec 루프 rolls size =
        if size < power + 1
        then 루프 (roll () :: rolls) (size + 1)
        else accumulate_dice_rolls rolls power 0 |> function
            | Some (result, _) when result <= max => result
            | _ when reroll => 루프 (listm.init power (fun _ => roll ())) power
            | _ => 루프 (roll () :: rolls) (size + 1)
    루프 [] 0

In [17]:
///- --test

roll_progressively roll_dice false 1
|> _assert_eq 1i32

16:36:24 1d dice.accumulate_dice_rolls / { power = 0; acc = 0; roll = 2; value = 1 }
16:36:24 2d dice.accumulate_dice_rolls / { power = -1; acc = 1; result = 2 }
16:36:24 3d dice.accumulate_dice_rolls / { power = 0; acc = 0; roll = 1 }
16:36:24 4d dice.accumulate_dice_rolls / { power = -1; acc = 0; result = 1 }
__assert_eq { expected = 1 }

In [18]:
///- --test

inl sequential_roll = create_sequential_roller [ 5; 4; 4; 5; 1 ]

roll_progressively sequential_roll false 2000i32
|> _assert_eq 995

16:36:25 1d dice.calculate_dice_count / { max = 2000; p = 7776; n = 5 }
16:36:25 2d dice.create_sequential_roller / roll / { current_index = 0; acc = 1; len = -1; last_item = null }
16:36:25 3d dice.create_sequential_roller / roll / { current_index = 1; acc = 1; len = -1; last_item = 5 }
16:36:25 4d dice.create_sequential_roller / roll / { current_index = 2; acc = 1; len = -1; last_item = 4 }
16:36:25 5d dice.create_sequential_roller / roll / { current_index = 3; acc = 1; len = -1; last_item = 4 }
16:36:25 6d dice.create_sequential_roller / roll / { current_index = 4; acc = 1; len = -1; last_item = 5 }
16:36:25 7d dice.accumulate_dice_rolls / { power = 4; acc = 0; roll = 1 }
16:36:25 8d dice.accumulate_dice_rolls / { power = 3; acc = 0; roll = 5; value = 864 }
16:36:25 9d dice.accumulate_dice_rolls / { power = 2; acc = 864; roll = 4; value = 108 }
16:36:25 10d dice.accumulate_dice_rolls / { power = 1; acc = 972; roll = 4; value = 18 }
16:36:25 11d dice.accumulate_dice_rolls / { power =

In [19]:
///- --test

inl sequential_roll = create_sequential_roller [ 5; 4; 4; 5; 2 ]

roll_progressively sequential_roll false 2000i32
|> _assert_eq 1678

16:36:25 1d dice.calculate_dice_count / { max = 2000; p = 7776; n = 5 }
16:36:25 2d dice.create_sequential_roller / roll / { current_index = 0; acc = 1; len = -1; last_item = null }
16:36:25 3d dice.create_sequential_roller / roll / { current_index = 1; acc = 1; len = -1; last_item = 5 }
16:36:25 4d dice.create_sequential_roller / roll / { current_index = 2; acc = 1; len = -1; last_item = 4 }
16:36:25 5d dice.create_sequential_roller / roll / { current_index = 3; acc = 1; len = -1; last_item = 4 }
16:36:25 6d dice.create_sequential_roller / roll / { current_index = 4; acc = 1; len = -1; last_item = 5 }
16:36:25 7d dice.accumulate_dice_rolls / { power = 4; acc = 0; roll = 2; value = 1296 }
16:36:25 8d dice.accumulate_dice_rolls / { power = 3; acc = 1296; roll = 5; value = 864 }
16:36:25 9d dice.accumulate_dice_rolls / { power = 2; acc = 2160; roll = 4; value = 108 }
16:36:25 10d dice.accumulate_dice_rolls / { power = 1; acc = 2268; roll = 4; value = 18 }
16:36:25 11d dice.accumulate_dic

In [20]:
///- --test

inl sequential_roll = create_sequential_roller [ 6u8; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6 ]

roll_progressively sequential_roll false 4738381338321616896u64
|> _assert_eq 4738381338321616896u64

16:36:28 1d dice.calculate_dice_count / { max = 4738381338321616896; p = 4738381338321616896; n = 24 }
16:36:28 2d dice.create_sequential_roller / roll / { current_index = 0; acc = 1; len = -1; last_item = null }
16:36:28 3d dice.create_sequential_roller / roll / { current_index = 1; acc = 1; len = -1; last_item = 6 }
16:36:28 4d dice.create_sequential_roller / roll / { current_index = 2; acc = 1; len = -1; last_item = 6 }
16:36:28 5d dice.create_sequential_roller / roll / { current_index = 3; acc = 1; len = -1; last_item = 6 }
16:36:28 6d dice.create_sequential_roller / roll / { current_index = 4; acc = 1; len = -1; last_item = 6 }
16:36:28 7d dice.create_sequential_roller / roll / { current_index = 5; acc = 1; len = -1; last_item = 6 }
16:36:28 8d dice.create_sequential_roller / roll / { current_index = 6; acc = 1; len = -1; last_item = 6 }
16:36:28 9d dice.create_sequential_roller / roll / { current_index = 7; acc = 1; len = -1; last_item = 6 }
16:36:28 10d dice.create_sequential_ro

In [21]:
///- --test

inl histogram (reroll : bool) : i32 =
    inl { level } = get_trace_state_or_init None
    level <- Info
    let rec insert (row : i32 * i32) (rows : list (i32 * i32)) : list (i32 * i32) =
        match rows with
        | [] => [ row ]
        | (k2, v2) :: rest =>
            inl k, v = row
            if k <= k2 then (k, v) :: (k2, v2) :: rest
            else (k2, v2) :: insert row rest
    join dyn listm'.init_series 1i32 10000 1
    |> listm.map (fun _ => roll_progressively roll_dice reroll 10i32)
    |> listm'.group_by id
    |> listm.map (fun (k, v) => k, (listm.length v : i32))
    |> listm.fold (fun acc row => insert row acc) []
    |> listm.fold (fun acc (k, v) => if acc = "" then #?k ++# " = " ++# #?v else acc ++# "; " ++# #?k ++# " = " ++# #?v) ""
    |> console.write_line
    0i32

histogram false

1 = 290; 2 = 710; 3 = 1341; 4 = 1355; 5 = 1314; 6 = 1309; 7 = 288; 8 = 765; 9 = 1338; 10 = 1290

In [22]:
///- --test

inl histogram (reroll : bool) : i32 =
    inl { level } = get_trace_state_or_init None
    level <- Info
    let rec insert (row : i32 * i32) (rows : list (i32 * i32)) : list (i32 * i32) =
        match rows with
        | [] => [ row ]
        | (k2, v2) :: rest =>
            inl k, v = row
            if k <= k2 then (k, v) :: (k2, v2) :: rest
            else (k2, v2) :: insert row rest
    join dyn listm'.init_series 1i32 10000 1
    |> listm.map (fun _ => roll_progressively roll_dice reroll 10i32)
    |> listm'.group_by id
    |> listm.map (fun (k, v) => k, (listm.length v : i32))
    |> listm.fold (fun acc row => insert row acc) []
    |> listm.fold (fun acc (k, v) => if acc = "" then #?k ++# " = " ++# #?v else acc ++# "; " ++# #?k ++# " = " ++# #?v) ""
    |> console.write_line
    0i32

histogram true

1 = 982; 2 = 976; 3 = 999; 4 = 966; 5 = 996; 6 = 1037; 7 = 1015; 8 = 993; 9 = 1039; 10 = 997

In [23]:
///- --test

inl check_progress () : i32 =
    inl { level } = get_trace_state_or_init None
    level <- Info
    join dyn listm'.init_series 1i32 100 1
    |> listm.iter fun n =>
        listm'.init_series 0i32 1 1
        |> listm.iter fun reroll =>
            join dyn listm'.init_series 1i32 3500 1
            |> listm.map fun _ => roll_progressively roll_dice (reroll = 1) n
            |> listm'.group_by id
            |> listm.length
            |> __assert_eq Silent n
    0i32

check_progress ()

## main

In [24]:
inl main (_args : array_base string) =
    inl result = roll_progressively roll_dice true 4738381338321616896i64
    trace Debug
        fun () => "dice.main"
        fun () => { result }
    0i32

inl main () =
    inl rotate_numbers' : i64 -> stream.stream u8 -> stream.stream u8 =
        rotate_numbers

    inl create_sequential_roller' :
        list u8
        -> (() -> u8)
        =
        create_sequential_roller

    inl roll_progressively' :
        (() -> u8)
        -> bool
        -> u64
        -> u64
        =
        roll_progressively

    inl roll_within_bounds' :
        u64
        -> list u8
        -> optionm'.option' u64
        =
        fun a b =>
            roll_within_bounds a b
            |> optionm'.box

    backend_switch {
        Fsharp = fun () => $'let rotate_numbers x = !rotate_numbers' x' : ()
        Rust = fun () => ()
        TypeScript = fun () => ()
    } : ()
    backend_switch {
        Fsharp = fun () => $'let create_sequential_roller x = !create_sequential_roller' x' : ()
        Rust = fun () => ()
        TypeScript = fun () => ()
    } : ()
    backend_switch {
        Fsharp = fun () => $'let roll_progressively x = !roll_progressively' x' : ()
        Rust = fun () => ()
        TypeScript = fun () => ()
    } : ()
    backend_switch {
        Fsharp = fun () => $'let roll_within_bounds x = !roll_within_bounds' x' : ()
        Rust = fun () => ()
        TypeScript = fun () => ()
    } : ()
    backend_switch {
        Fsharp = fun () => $'let main args = !main args' : ()
        Rust = fun () => main ;[] |> ignore
        Python = fun () => main ;[] |> ignore
        TypeScript = fun () => main ;[] |> ignore
    } : ()